In [1]:
# ============================================================
# AI Medical Diagnosis by Dr. Arpit Yadav
# Utility-Based Agent using LangGraph + Groq + Serper + Gradio
# Educational Demo Only - Not for real clinical use
# ============================================================

# =========================
# 1. Install dependencies
# =========================
!pip -q install langgraph langchain langchain-groq gradio requests pandas pydantic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 15.6 MB/s eta 0:00:00


In [2]:
# =========================
# 2. Import all necessary libraries
# =========================
import os
import json
import math
import requests
import pandas as pd
from typing import TypedDict, List, Dict, Any

import gradio as gr

from langgraph.graph import StateGraph, END
from langchain_groq import ChatGroq

In [3]:


# =========================
# 3. Give Groq API and Serper API
# =========================
# Option 1:
# os.environ["GROQ_API_KEY"] = "your_groq_api_key"
# os.environ["SERPER_API_KEY"] = "your_serper_api_key"

if "GROQ_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["GROQ_API_KEY"] = getpass("Enter GROQ_API_KEY: ")
    except Exception:
        pass

if "SERPER_API_KEY" not in os.environ:
    try:
        from getpass import getpass
        os.environ["SERPER_API_KEY"] = getpass("Enter SERPER_API_KEY: ")
    except Exception:
        pass

GROQ_API_KEY = os.getenv("GROQ_API_KEY", "")
SERPER_API_KEY = os.getenv("SERPER_API_KEY", "")

Enter GROQ_API_KEY: ··········
Enter SERPER_API_KEY: ··········


In [4]:
# =========================
# 4. Select Model from Groq : use llama-3.1-8b-instant
# =========================
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key=GROQ_API_KEY
)

In [5]:



# ============================================================
# Utility-Based Agent Logic
# The agent scores treatment plans using utility:
# expected_outcome vs patient_risk vs side_effect_probability vs cost
# and selects the best overall utility plan.
# ============================================================

# =========================
# 5. Sample treatment knowledge base (educational demo)
# =========================
# Scores are synthetic and simplified for teaching purposes.

TREATMENT_KB = {
    "Hypertension": [
        {
            "plan_id": "HTN_P1",
            "plan_name": "Lifestyle-first Plan",
            "category": "Non-Pharmacological",
            "base_expected_outcome": 60,
            "base_risk": 10,
            "base_side_effect": 5,
            "base_cost": 15,
            "monitoring_need": 20,
            "best_for": "mild"
        },
        {
            "plan_id": "HTN_P2",
            "plan_name": "Standard Single Medication Plan",
            "category": "Medication",
            "base_expected_outcome": 78,
            "base_risk": 18,
            "base_side_effect": 20,
            "base_cost": 40,
            "monitoring_need": 35,
            "best_for": "moderate"
        },
        {
            "plan_id": "HTN_P3",
            "plan_name": "Combination Therapy Plan",
            "category": "Medication",
            "base_expected_outcome": 88,
            "base_risk": 28,
            "base_side_effect": 30,
            "base_cost": 65,
            "monitoring_need": 45,
            "best_for": "severe"
        },
        {
            "plan_id": "HTN_P4",
            "plan_name": "Specialist Referral + Monitoring",
            "category": "Referral",
            "base_expected_outcome": 82,
            "base_risk": 16,
            "base_side_effect": 8,
            "base_cost": 75,
            "monitoring_need": 70,
            "best_for": "high-risk"
        }
    ],

    "Type 2 Diabetes": [
        {
            "plan_id": "DM_P1",
            "plan_name": "Lifestyle + Diet Plan",
            "category": "Non-Pharmacological",
            "base_expected_outcome": 62,
            "base_risk": 10,
            "base_side_effect": 5,
            "base_cost": 20,
            "monitoring_need": 25,
            "best_for": "mild"
        },
        {
            "plan_id": "DM_P2",
            "plan_name": "Standard Oral Medication Plan",
            "category": "Medication",
            "base_expected_outcome": 80,
            "base_risk": 18,
            "base_side_effect": 18,
            "base_cost": 45,
            "monitoring_need": 35,
            "best_for": "moderate"
        },
        {
            "plan_id": "DM_P3",
            "plan_name": "Intensified Combination Plan",
            "category": "Medication",
            "base_expected_outcome": 90,
            "base_risk": 30,
            "base_side_effect": 28,
            "base_cost": 70,
            "monitoring_need": 50,
            "best_for": "severe"
        },
        {
            "plan_id": "DM_P4",
            "plan_name": "Endocrinology Referral + Monitoring",
            "category": "Referral",
            "base_expected_outcome": 84,
            "base_risk": 14,
            "base_side_effect": 10,
            "base_cost": 80,
            "monitoring_need": 72,
            "best_for": "high-risk"
        }
    ],

    "Respiratory Infection (Suspected)": [
        {
            "plan_id": "RI_P1",
            "plan_name": "Supportive Care Plan",
            "category": "Supportive",
            "base_expected_outcome": 58,
            "base_risk": 8,
            "base_side_effect": 4,
            "base_cost": 10,
            "monitoring_need": 20,
            "best_for": "mild"
        },
        {
            "plan_id": "RI_P2",
            "plan_name": "Primary Care Medication Review",
            "category": "Medication Review",
            "base_expected_outcome": 74,
            "base_risk": 16,
            "base_side_effect": 16,
            "base_cost": 35,
            "monitoring_need": 30,
            "best_for": "moderate"
        },
        {
            "plan_id": "RI_P3",
            "plan_name": "Escalated Care + Diagnostics",
            "category": "Diagnostics",
            "base_expected_outcome": 86,
            "base_risk": 20,
            "base_side_effect": 12,
            "base_cost": 68,
            "monitoring_need": 55,
            "best_for": "severe"
        },
        {
            "plan_id": "RI_P4",
            "plan_name": "Urgent Specialist / Hospital Evaluation",
            "category": "Referral",
            "base_expected_outcome": 90,
            "base_risk": 14,
            "base_side_effect": 8,
            "base_cost": 88,
            "monitoring_need": 80,
            "best_for": "high-risk"
        }
    ]
}


In [6]:


# =========================
# 6. Symptoms / rule-based condition inference
# =========================
CONDITION_RULES = {
    "Hypertension": {
        "symptoms": ["Headache", "Dizziness", "Blurred Vision", "Fatigue"],
        "high_bp_bonus": 25
    },
    "Type 2 Diabetes": {
        "symptoms": ["Frequent Urination", "Excessive Thirst", "Fatigue", "Blurred Vision"],
        "high_glucose_bonus": 25
    },
    "Respiratory Infection (Suspected)": {
        "symptoms": ["Fever", "Cough", "Shortness of Breath", "Chest Discomfort", "Fatigue"],
        "high_temp_bonus": 25
    }
}

In [7]:

# =========================
# 7. LangGraph state
# =========================
class MedState(TypedDict, total=False):
    age: int
    gender: str
    pregnancy_status: str
    symptoms: List[str]
    severity: str
    systolic_bp: float
    glucose_level: float
    temperature_c: float
    kidney_risk: str
    liver_risk: str
    allergy_sensitivity: str
    budget_sensitivity: str
    side_effect_tolerance: str
    expected_outcome_priority: str
    search_mode: str
    manual_condition: str

    inferred_condition: str
    candidate_plans: List[Dict[str, Any]]
    web_evidence: str
    scored_plans: List[Dict[str, Any]]
    selected_plan: Dict[str, Any]
    explanation: str
    final_report: str

In [8]:



# =========================
# 8. Helper functions
# =========================
def clamp(x, low=0, high=100):
    return max(low, min(high, x))

def normalize(values):
    min_v = min(values)
    max_v = max(values)
    if max_v == min_v:
        return [0.5] * len(values)
    return [(v - min_v) / (max_v - min_v) for v in values]

def severity_to_multiplier(severity: str) -> float:
    return {
        "Mild": 0.90,
        "Moderate": 1.00,
        "High": 1.10,
        "Severe": 1.20
    }.get(severity, 1.00)

def preference_to_weight(level: str, mapping_type="positive") -> float:
    # positive means higher preference -> larger weight on maximizing outcome
    if mapping_type == "positive":
        return {"Low": 0.8, "Medium": 1.0, "High": 1.2}.get(level, 1.0)
    # negative means higher concern -> larger penalty on risk/side effects/cost
    return {"Low": 0.8, "Medium": 1.0, "High": 1.2}.get(level, 1.0)

def serper_search(query: str, num_results: int = 3) -> str:
    if not SERPER_API_KEY:
        return "Serper API key not available."

    url = "https://google.serper.dev/search"
    payload = json.dumps({"q": query, "num": num_results})
    headers = {
        "X-API-KEY": SERPER_API_KEY,
        "Content-Type": "application/json"
    }

    try:
        response = requests.post(url, headers=headers, data=payload, timeout=20)
        response.raise_for_status()
        data = response.json()

        snippets = []
        for item in data.get("organic", [])[:num_results]:
            title = item.get("title", "")
            snippet = item.get("snippet", "")
            link = item.get("link", "")
            snippets.append(f"- {title} | {link}\n  {snippet}")

        if not snippets:
            return "No notable public web evidence found."
        return "\n".join(snippets)
    except Exception as e:
        return f"Serper search failed: {str(e)}"

In [9]:

# =========================
# 9. LangGraph nodes
# =========================
def infer_condition_node(state: MedState) -> MedState:
    if state.get("manual_condition") and state["manual_condition"] != "Auto Detect":
        return {**state, "inferred_condition": state["manual_condition"]}

    symptoms = set(state.get("symptoms", []))
    systolic_bp = float(state.get("systolic_bp", 120))
    glucose = float(state.get("glucose_level", 100))
    temp = float(state.get("temperature_c", 37.0))

    scores = {
        "Hypertension": 0,
        "Type 2 Diabetes": 0,
        "Respiratory Infection (Suspected)": 0
    }

    for condition, cfg in CONDITION_RULES.items():
        for s in cfg["symptoms"]:
            if s in symptoms:
                scores[condition] += 12

    if systolic_bp >= 140:
        scores["Hypertension"] += CONDITION_RULES["Hypertension"]["high_bp_bonus"]

    if glucose >= 180:
        scores["Type 2 Diabetes"] += CONDITION_RULES["Type 2 Diabetes"]["high_glucose_bonus"]

    if temp >= 38.0:
        scores["Respiratory Infection (Suspected)"] += CONDITION_RULES["Respiratory Infection (Suspected)"]["high_temp_bonus"]

    inferred = max(scores, key=scores.get)
    return {**state, "inferred_condition": inferred}


def fetch_candidate_plans_node(state: MedState) -> MedState:
    condition = state.get("inferred_condition", "Hypertension")
    plans = TREATMENT_KB.get(condition, [])
    return {**state, "candidate_plans": plans}


def optional_web_search_node(state: MedState) -> MedState:
    search_mode = state.get("search_mode", "Auto")
    condition = state.get("inferred_condition", "")

    if search_mode == "Off":
        return {**state, "web_evidence": "Web search disabled."}

    if search_mode in ["On", "Auto"]:
        query = f"general overview of treatment options and monitoring for {condition}"
        evidence = serper_search(query)
        return {**state, "web_evidence": evidence}

    return {**state, "web_evidence": "No web evidence used."}


def utility_scoring_node(state: MedState) -> MedState:
    plans = state.get("candidate_plans", [])
    if not plans:
        return {**state, "scored_plans": [], "selected_plan": {}}

    age = int(state.get("age", 40))
    severity = state.get("severity", "Moderate")
    kidney_risk = state.get("kidney_risk", "Low")
    liver_risk = state.get("liver_risk", "Low")
    allergy_sensitivity = state.get("allergy_sensitivity", "Low")
    budget_sensitivity = state.get("budget_sensitivity", "Medium")
    side_effect_tolerance = state.get("side_effect_tolerance", "Medium")
    outcome_priority = state.get("expected_outcome_priority", "High")
    pregnancy_status = state.get("pregnancy_status", "No")

    sev_mult = severity_to_multiplier(severity)

    adjusted = []
    for p in plans:
        expected_outcome = p["base_expected_outcome"] * sev_mult
        risk = p["base_risk"]
        side_effect = p["base_side_effect"]
        cost = p["base_cost"]
        monitoring = p["monitoring_need"]

        # patient-specific adjustments
        if age >= 65:
            risk += 8
            side_effect += 6
            monitoring += 8

        if kidney_risk == "High":
            risk += 10
            side_effect += 6

        if liver_risk == "High":
            risk += 8
            side_effect += 6

        if allergy_sensitivity == "High" and p["category"] == "Medication":
            side_effect += 10
            risk += 4

        if pregnancy_status == "Yes" and p["category"] == "Medication":
            risk += 12
            side_effect += 6

        if p["best_for"] == "mild" and severity in ["High", "Severe"]:
            expected_outcome -= 12

        if p["best_for"] == "severe" and severity == "Mild":
            cost += 8
            risk += 4

        if p["best_for"] == "high-risk" and age < 50 and kidney_risk == "Low" and liver_risk == "Low":
            cost += 10

        adjusted.append({
            **p,
            "adj_expected_outcome": clamp(expected_outcome),
            "adj_risk": clamp(risk),
            "adj_side_effect": clamp(side_effect),
            "adj_cost": clamp(cost),
            "adj_monitoring": clamp(monitoring)
        })

    outcomes = [x["adj_expected_outcome"] for x in adjusted]
    risks = [x["adj_risk"] for x in adjusted]
    sides = [x["adj_side_effect"] for x in adjusted]
    costs = [x["adj_cost"] for x in adjusted]
    monitoring_scores = [x["adj_monitoring"] for x in adjusted]

    norm_outcomes = normalize(outcomes)
    norm_risks = normalize(risks)
    norm_sides = normalize(sides)
    norm_costs = normalize(costs)
    norm_monitor = normalize(monitoring_scores)

    outcome_weight = preference_to_weight(outcome_priority, "positive") * 0.40
    risk_weight = preference_to_weight("High" if severity in ["High", "Severe"] else "Medium", "negative") * 0.25
    side_weight = preference_to_weight("High" if side_effect_tolerance == "Low" else "Medium", "negative") * 0.20
    cost_weight = preference_to_weight(budget_sensitivity, "negative") * 0.15

    scored_plans = []
    for i, p in enumerate(adjusted):
        utility = (
            norm_outcomes[i] * outcome_weight
            - norm_risks[i] * risk_weight
            - norm_sides[i] * side_weight
            - norm_costs[i] * cost_weight
            - norm_monitor[i] * 0.05
        )

        scored_plans.append({
            **p,
            "utility_score": round(utility, 4)
        })

    scored_plans = sorted(scored_plans, key=lambda x: x["utility_score"], reverse=True)
    selected_plan = scored_plans[0]

    return {
        **state,
        "scored_plans": scored_plans,
        "selected_plan": selected_plan
    }


def llm_explainer_node(state: MedState) -> MedState:
    selected_plan = state.get("selected_plan", {})
    condition = state.get("inferred_condition", "")
    web_evidence = state.get("web_evidence", "")
    scored_plans = state.get("scored_plans", [])

    if not selected_plan:
        return {
            **state,
            "explanation": "No plan could be selected."
        }

    comparison = "\n".join([
        f"- {p['plan_id']} | {p['plan_name']} | outcome={p['adj_expected_outcome']} | risk={p['adj_risk']} | side_effect={p['adj_side_effect']} | cost={p['adj_cost']} | utility={p['utility_score']}"
        for p in scored_plans[:3]
    ])

    prompt = f"""
You are a medical AI product assistant for an educational demo.

Do NOT provide a real diagnosis, prescription, or medical advice.
The utility-based planner has already selected the plan.
Your task is only to explain the result clearly and responsibly.

Condition inferred: {condition}

Selected plan:
{json.dumps(selected_plan, indent=2)}

Top plan comparison:
{comparison}

Public web context:
{web_evidence}

Write:
1. A short executive explanation
2. Why this plan scored highest in utility
3. Key trade-offs
4. A safety note saying clinical validation is required

Keep it concise, professional, and product-style.
"""

    try:
        resp = llm.invoke(prompt)
        explanation = resp.content if hasattr(resp, "content") else str(resp)
    except Exception as e:
        explanation = f"LLM explanation unavailable: {str(e)}"

    return {**state, "explanation": explanation}


def formatter_node(state: MedState) -> MedState:
    selected = state.get("selected_plan", {})
    scored = state.get("scored_plans", [])
    condition = state.get("inferred_condition", "")
    explanation = state.get("explanation", "")
    web_evidence = state.get("web_evidence", "")

    if not selected:
        report = """
# AI Medical Diagnosis

## Result
No valid treatment plan could be selected.
This demo is educational only and must not be used for real clinical decisions.
        """.strip()
        return {**state, "final_report": report}

    ranking_lines = []
    for idx, p in enumerate(scored, start=1):
        ranking_lines.append(
            f"{idx}. {p['plan_id']} | {p['plan_name']} | Outcome: {p['adj_expected_outcome']} | "
            f"Risk: {p['adj_risk']} | Side Effect: {p['adj_side_effect']} | "
            f"Cost: {p['adj_cost']} | Utility: {p['utility_score']}"
        )

    report = f"""
# AI Medical Diagnosis

## Inferred Condition
**Condition:** {condition}

## Recommended Plan
**Plan ID:** {selected.get('plan_id')}
**Plan Name:** {selected.get('plan_name')}
**Category:** {selected.get('category')}
**Expected Outcome Score:** {selected.get('adj_expected_outcome')}
**Patient Risk Score:** {selected.get('adj_risk')}
**Side Effect Probability Score:** {selected.get('adj_side_effect')}
**Cost Score:** {selected.get('adj_cost')}
**Monitoring Need Score:** {selected.get('adj_monitoring')}
**Utility Score:** {selected.get('utility_score')}

## Plan Ranking
{chr(10).join([f"- {x}" for x in ranking_lines])}

## Public Web Evidence
{web_evidence}

## Explanation
{explanation}

## Safety Note
This is an educational utility-based AI demo only.
It must not be used as a real diagnosis, prescription, or emergency decision system.
Please consult a qualified clinician.
    """.strip()

    return {**state, "final_report": report}

In [10]:

# =========================
# 10. Build LangGraph
# =========================
builder = StateGraph(MedState)

builder.add_node("infer_condition", infer_condition_node)
builder.add_node("fetch_candidate_plans", fetch_candidate_plans_node)
builder.add_node("optional_web_search", optional_web_search_node)
builder.add_node("utility_scoring", utility_scoring_node)
builder.add_node("llm_explainer", llm_explainer_node)
builder.add_node("formatter", formatter_node)

builder.set_entry_point("infer_condition")
builder.add_edge("infer_condition", "fetch_candidate_plans")
builder.add_edge("fetch_candidate_plans", "optional_web_search")
builder.add_edge("optional_web_search", "utility_scoring")
builder.add_edge("utility_scoring", "llm_explainer")
builder.add_edge("llm_explainer", "formatter")
builder.add_edge("formatter", END)

medical_graph = builder.compile()

In [11]:



# =========================
# 11. Main function
# =========================
def run_medical_utility_agent(
    age, gender, pregnancy_status, symptoms, severity,
    systolic_bp, glucose_level, temperature_c,
    kidney_risk, liver_risk, allergy_sensitivity,
    budget_sensitivity, side_effect_tolerance,
    expected_outcome_priority, search_mode, manual_condition
):
    state = {
        "age": int(age),
        "gender": gender,
        "pregnancy_status": pregnancy_status,
        "symptoms": symptoms or [],
        "severity": severity,
        "systolic_bp": float(systolic_bp),
        "glucose_level": float(glucose_level),
        "temperature_c": float(temperature_c),
        "kidney_risk": kidney_risk,
        "liver_risk": liver_risk,
        "allergy_sensitivity": allergy_sensitivity,
        "budget_sensitivity": budget_sensitivity,
        "side_effect_tolerance": side_effect_tolerance,
        "expected_outcome_priority": expected_outcome_priority,
        "search_mode": search_mode,
        "manual_condition": manual_condition
    }

    result = medical_graph.invoke(state)

    selected = result.get("selected_plan", {})
    scored_plans = result.get("scored_plans", [])
    final_report = result.get("final_report", "")
    inferred_condition = result.get("inferred_condition", "N/A")

    if scored_plans:
        df = pd.DataFrame(scored_plans)[[
            "plan_id", "plan_name", "category",
            "adj_expected_outcome", "adj_risk",
            "adj_side_effect", "adj_cost",
            "adj_monitoring", "utility_score"
        ]]
    else:
        df = pd.DataFrame(columns=[
            "plan_id", "plan_name", "category",
            "adj_expected_outcome", "adj_risk",
            "adj_side_effect", "adj_cost",
            "adj_monitoring", "utility_score"
        ])

    selected_plan_name = selected.get("plan_name", "N/A")
    utility_score = selected.get("utility_score", 0)
    outcome_score = selected.get("adj_expected_outcome", 0)
    risk_score = selected.get("adj_risk", 0)

    return final_report, inferred_condition, selected_plan_name, utility_score, outcome_score, risk_score, df

In [12]:



# =========================
# 12. Examples
# =========================
EXAMPLES = {
    "Hypertension Case": (
        58, "Male", "No",
        ["Headache", "Dizziness", "Blurred Vision"],
        "High", 152, 105, 36.9,
        "Low", "Low", "Medium",
        "Medium", "Low", "High",
        "Auto", "Auto Detect"
    ),
    "Diabetes Case": (
        49, "Female", "No",
        ["Frequent Urination", "Excessive Thirst", "Fatigue"],
        "Moderate", 128, 210, 36.8,
        "Low", "Low", "Low",
        "High", "Medium", "High",
        "Auto", "Auto Detect"
    ),
    "Respiratory Case": (
        67, "Female", "No",
        ["Fever", "Cough", "Shortness of Breath", "Fatigue"],
        "Severe", 132, 118, 38.6,
        "High", "Low", "High",
        "Medium", "Low", "High",
        "Auto", "Auto Detect"
    )
}

def load_example(example_name):
    return EXAMPLES[example_name]

In [ ]:



# =========================
# 13. Professional Gradio UI
# =========================
CUSTOM_CSS = """
body {
    background: linear-gradient(135deg, #f8fbff, #eefaf3);
}

.gradio-container {
    max-width: 1480px !important;
    margin: auto;
}

.hero-wrap {
    border-radius: 18px;
    padding: 18px 22px;
    background: linear-gradient(90deg, #eef7ff, #f6fff8);
    border: 1px solid #d9e8ff;
    box-shadow: 0 8px 24px rgba(0,0,0,0.06);
    overflow: hidden;
}

.hero-title {
    font-size: 34px;
    font-weight: 800;
    color: #184e77;
    margin-bottom: 4px;
}

.hero-subtitle {
    font-size: 14px;
    color: #4d6b86;
    margin-bottom: 12px;
}

.marquee {
    width: 100%;
    overflow: hidden;
    white-space: nowrap;
    border-radius: 10px;
    background: linear-gradient(90deg, #dff6e8, #e6f0ff);
    border: 1px solid #cfe3ff;
    padding: 10px 0;
}

.marquee span {
    display: inline-block;
    padding-left: 100%;
    animation: marquee 18s linear infinite;
    color: #0b6e4f;
    font-weight: 700;
    font-size: 18px;
}

@keyframes marquee {
    0%   { transform: translateX(0%); }
    100% { transform: translateX(-100%); }
}

.footer-note {
    color: #5b7083;
    font-size: 13px;
}

.warning-box {
    background: #fff4e5;
    border: 1px solid #ffd59e;
    color: #7a4b00;
    padding: 12px 14px;
    border-radius: 12px;
    margin-bottom: 14px;
    font-size: 14px;
}
"""

theme = gr.themes.Soft(
    primary_hue="blue",
    secondary_hue="green",
    neutral_hue="slate"
)

with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Medical Diagnosis") as demo:

    gr.HTML("""
    <div class="hero-wrap">
        <div class="hero-title">AI Medical Diagnosis</div>
        <div class="hero-subtitle">Utility-Based Agent using LangGraph + Groq + Serper</div>
        <div class="marquee">
            <span>Utility-Based Agent AI Medical Diagnosis by Dr. Arpit Yadav • Utility-Based Agent AI Medical Diagnosis by Dr. Arpit Yadav • Utility-Based Agent AI Medical Diagnosis by Dr. Arpit Yadav</span>
        </div>
    </div>
    """)

    gr.HTML("""
    <div class="warning-box">
        <b>Important:</b> This is an educational simulation only. It is not a medical device, not a diagnosis engine,
        and not a substitute for professional care.
    </div>
    """)

    with gr.Row():
        with gr.Column(scale=5):
            gr.Markdown("### Patient Input")

            example_choice = gr.Dropdown(
                choices=list(EXAMPLES.keys()),
                value="Hypertension Case",
                label="Load Example Case",
                interactive=True
            )

            with gr.Row():
                age = gr.Number(value=58, label="Age")
                gender = gr.Dropdown(
                    choices=["Male", "Female", "Other"],
                    value="Male",
                    label="Gender"
                )
                pregnancy_status = gr.Dropdown(
                    choices=["No", "Yes", "Not Applicable"],
                    value="No",
                    label="Pregnancy Status"
                )

            symptoms = gr.CheckboxGroup(
                choices=[
                    "Headache", "Dizziness", "Blurred Vision", "Fatigue",
                    "Frequent Urination", "Excessive Thirst",
                    "Fever", "Cough", "Shortness of Breath", "Chest Discomfort"
                ],
                value=["Headache", "Dizziness"],
                label="Symptoms"
            )

            with gr.Row():
                severity = gr.Dropdown(
                    choices=["Mild", "Moderate", "High", "Severe"],
                    value="High",
                    label="Severity"
                )
                manual_condition = gr.Dropdown(
                    choices=["Auto Detect", "Hypertension", "Type 2 Diabetes", "Respiratory Infection (Suspected)"],
                    value="Auto Detect",
                    label="Condition Selection"
                )

            with gr.Row():
                systolic_bp = gr.Number(value=152, label="Systolic BP")
                glucose_level = gr.Number(value=105, label="Glucose Level")
                temperature_c = gr.Number(value=36.9, label="Temperature (°C)")

        with gr.Column(scale=3):
            gr.Markdown("### Risk & Preference Settings")

            with gr.Row():
                kidney_risk = gr.Dropdown(["Low", "Medium", "High"], value="Low", label="Kidney Risk")
                liver_risk = gr.Dropdown(["Low", "Medium", "High"], value="Low", label="Liver Risk")

            with gr.Row():
                allergy_sensitivity = gr.Dropdown(["Low", "Medium", "High"], value="Medium", label="Allergy Sensitivity")
                budget_sensitivity = gr.Dropdown(["Low", "Medium", "High"], value="Medium", label="Budget Sensitivity")

            with gr.Row():
                side_effect_tolerance = gr.Dropdown(["Low", "Medium", "High"], value="Low", label="Side Effect Tolerance")
                expected_outcome_priority = gr.Dropdown(["Low", "Medium", "High"], value="High", label="Expected Outcome Priority")

            search_mode = gr.Dropdown(
                choices=["Auto", "On", "Off"],
                value="Auto",
                label="Web Search Enrichment"
            )

            run_btn = gr.Button("Run AI Medical Diagnosis", variant="primary")
            clear_btn = gr.Button("Clear", variant="secondary")

            gr.Markdown("""
            **How this works**
            - Infers likely condition using simple rules
            - Fetches candidate treatment plans
            - Scores plans using utility
            - Uses Groq for explanation only
            - Optionally enriches with Serper
            """)

    with gr.Row():
        inferred_condition = gr.Textbox(label="Inferred Condition")
        selected_plan_name = gr.Textbox(label="Selected Plan")
        utility_score = gr.Number(label="Utility Score")
        outcome_score = gr.Number(label="Outcome Score")
        risk_score = gr.Number(label="Risk Score")

    with gr.Row():
        report = gr.Markdown(label="Medical Utility Report")
        plans_table = gr.Dataframe(label="Ranked Treatment Plans", interactive=False)

    example_choice.change(
        fn=load_example,
        inputs=example_choice,
        outputs=[
            age, gender, pregnancy_status, symptoms, severity,
            systolic_bp, glucose_level, temperature_c,
            kidney_risk, liver_risk, allergy_sensitivity,
            budget_sensitivity, side_effect_tolerance,
            expected_outcome_priority, search_mode, manual_condition
        ]
    )

    run_btn.click(
        fn=run_medical_utility_agent,
        inputs=[
            age, gender, pregnancy_status, symptoms, severity,
            systolic_bp, glucose_level, temperature_c,
            kidney_risk, liver_risk, allergy_sensitivity,
            budget_sensitivity, side_effect_tolerance,
            expected_outcome_priority, search_mode, manual_condition
        ],
        outputs=[
            report, inferred_condition, selected_plan_name,
            utility_score, outcome_score, risk_score, plans_table
        ]
    )

    clear_btn.click(
        fn=lambda: (
            "", "", "", 0, 0, 0,
            pd.DataFrame(columns=[
                "plan_id", "plan_name", "category",
                "adj_expected_outcome", "adj_risk",
                "adj_side_effect", "adj_cost",
                "adj_monitoring", "utility_score"
            ])
        ),
        inputs=[],
        outputs=[
            report, inferred_condition, selected_plan_name,
            utility_score, outcome_score, risk_score, plans_table
        ]
    )

    gr.Markdown(
        "<div class='footer-note'>Professional product-style educational demo • Utility-Based Agent • LangGraph workflow • Groq explanation layer • Serper enrichment</div>"
    )

demo.launch(debug=True, share=True)

/tmp/ipykernel_2170/2168488775.py:82: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Medical Diagnosis") as demo:
/tmp/ipykernel_2170/2168488775.py:82: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=theme, css=CUSTOM_CSS, title="AI Medical Diagnosis") as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://a90c05753e354062c0.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
